In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [7]:
df = pd.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [188]:
df.columns

Index(['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI',
       'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [201]:
class Normalization(object):
    def __init__(self, dataset):
        self.dataset = dataset.reset_index(drop = True)
        self.df_max_scaled = dataset.copy()
        self.df_min_max_scaled = dataset.copy()
        self.z_scaled = dataset.copy()
        self.column = ['HighBP', 'HighChol', 'CholCheck', 'BMI',
       'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income']
        
    def df_basic(self, dataset):
        global df_basic_global
        
        self.df_basic = pd.DataFrame(index = self.dataset.columns, columns=['Min', 'Max', 'Sum', 'Mean', 'Mode'])
        self.df_basic['Min'] = self.dataset.min()
        self.df_basic['Max'] = self.dataset.max()
        self.df_basic['Sum'] = self.dataset.sum()
        self.df_basic['Mean'] = self.dataset.mean()
        self.df_basic['Mode'] = self.dataset.mode().T
        
        df_basic_global = self.df_basic

    def max_scaled(self):
        global df_max_scaled_global

        self.df_max_scaled[self.column] = self.df_max_scaled[self.column] / self.df_max_scaled[self.column].abs().max()

        df_max_scaled_global = self.df_max_scaled

    def min_max_scaled(self):
        global df_min_max_scaled_global
        
        self.df_min_max_scaled[self.column] = (self.df_min_max_scaled[self.column] - self.df_min_max_scaled[self.column].min()) / (self.df_min_max_scaled[self.column].max()  - self.df_min_max_scaled[self.column].min())
        df_min_max_scaled_global = self.df_min_max_scaled

    def z_scaled_class(self):
        global z_scaled_global

        self.z_scaled[self.column] = (self.z_scaled[self.column] - self.z_scaled[self.column].mean()) / self.z_scaled[self.column].std()

        z_scaled_global = self.z_scaled

    def main(self):
        self.df_basic(self.dataset)
        self.max_scaled()
        self.min_max_scaled()
        self.z_scaled_class()

In [202]:
norm = Normalization(df)

In [203]:
norm.main()

In [204]:
class Classification(object):
    def __init__(self, dataset, label, test_size, random_state, shuffle, stratify, max_iter_LR, random_state_LR):
        self.dataset = dataset.reset_index(drop = True)
        self.X = self.dataset.copy().drop([label], axis = 1)
        self.y = self.dataset[[label]].copy()
        self.label = label
        self.test_size = test_size
        self.random_state = random_state
        self.shuffle = shuffle
        self.stratify = stratify
        self.max_iter_LR = max_iter_LR
        self.random_state_LR = random_state_LR

        

    def train_test_split_def(self):

        global X_train, X_test, y_train, y_test
        
        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y , test_size=self.test_size, random_state=self.random_state)


    def LogisticRegression(self):
        
        
        model = LogisticRegression(max_iter = self.max_iter_LR, random_state =  self.random_state_LR)
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)
        
        cm = confusion_matrix(y_test, predictions)
        
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("Logistic Regression's confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)

        acc = accuracy_score(y_test, predictions) * 100
        print(f"Logistic Regression model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"Logistic Regression model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"Logistic Regression model recall: {recall:.2f}%")
        print('\n')

    def LinearSVC(self):
        
        model = LinearSVC(random_state = 0, tol = 1e-5)
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)

        cm = confusion_matrix(y_test, predictions)
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("Linear Support Vector Classification's confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)
        
        acc = accuracy_score(y_test, predictions) * 100
        print(f"Linear Support Vector Classification model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"Linear Support Vector Classification model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"Linear Support Vector Classification model recall: {recall:.2f}%")
        print('\n')


    def Decision_tree(self):
        model = DecisionTreeClassifier()
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)

        cm = confusion_matrix(y_test, predictions)
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("Decision Tree Classifier's confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)
        
        acc = accuracy_score(y_test, predictions) * 100
        print(f"Decision Tree Classifier model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"Decision Tree Classifier model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"Decision Tree Classifier model recall: {recall:.2f}%")
        print('\n')

    def Random_forest(self):
        
        model = RandomForestClassifier(n_estimators = 10)
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)

        cm = confusion_matrix(y_test, predictions)
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("Random Forest Classifier's confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)
        
        acc = accuracy_score(y_test, predictions) * 100
        print(f"Random Forest Classifier model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"Random Forest Classifier model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"Random Forest Classifier model recall: {recall:.2f}%")
        print('\n')

    def Naive_Bayes(self):

        model = GaussianNB()
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)

        cm = confusion_matrix(y_test, predictions)
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("Naive Bayes's confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)
        
        acc = accuracy_score(y_test, predictions) * 100
        print(f"Naive Bayes model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"Naive Bayes model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"Naive Bayes model recall: {recall:.2f}%")
        print('\n')

    def K_Nearest_Neighbours(self):

        model = KNeighborsClassifier()
        model.fit(X_train, np.ravel(y_train))

        predictions = model.predict(X_test)

        cm = confusion_matrix(y_test, predictions)
        TN, FP, FN, TP = confusion_matrix(y_test, predictions).ravel()

        print("K Nearest Neighbours' confusion matrix: ")
        print('True Positive(TP)  = ', TP)
        print('False Positive(FP) = ', FP)
        print('True Negative(TN)  = ', TN)
        print('False Negative(FN) = ', FN)
        
        acc = accuracy_score(y_test, predictions) * 100
        print(f"K Nearest Neighbours model accuracy: {acc:.2f}%")
        precision = precision_score(predictions, y_test) * 100
        print(f"K Nearest Neighbours model precision: {precision:.2f}%")
        recall = recall_score(predictions, y_test) * 100
        print(f"K Nearest Neighbours model recall: {recall:.2f}%")
        print('\n')
        
    def main(self):
        self.train_test_split_def()
        self.LogisticRegression()
        self.LinearSVC()
        self.Decision_tree()
        self.Random_forest()
        self.Naive_Bayes()
        self.K_Nearest_Neighbours()
        

In [205]:
LR = Classification(df, 'HeartDiseaseorAttack', 0.20, 0, True, None, 10000, 0)

In [206]:
LR.main()

Logistic Regression's confusion matrix: 
True Positive(TP)  =  688
False Positive(FP) =  509
True Negative(TN)  =  45455
False Negative(FN) =  4084
Logistic Regression model accuracy: 90.95%
Logistic Regression model precision: 14.42%
Logistic Regression model recall: 57.48%


Linear Support Vector Classification's confusion matrix: 
True Positive(TP)  =  254
False Positive(FP) =  120
True Negative(TN)  =  45844
False Negative(FN) =  4518
Linear Support Vector Classification model accuracy: 90.86%
Linear Support Vector Classification model precision: 5.32%
Linear Support Vector Classification model recall: 67.91%


Decision Tree Classifier's confusion matrix: 
True Positive(TP)  =  1360
False Positive(FP) =  4132
True Negative(TN)  =  41832
False Negative(FN) =  3412
Decision Tree Classifier model accuracy: 85.13%
Decision Tree Classifier model precision: 28.50%
Decision Tree Classifier model recall: 24.76%


Random Forest Classifier's confusion matrix: 
True Positive(TP)  =  606
False

In [207]:
LR_1 = Classification(df_max_scaled_global, 'HeartDiseaseorAttack', 0.20, 0, True, None, 10000, 0)

In [208]:
LR_1.main()

Logistic Regression's confusion matrix: 
True Positive(TP)  =  690
False Positive(FP) =  507
True Negative(TN)  =  45457
False Negative(FN) =  4082
Logistic Regression model accuracy: 90.96%
Logistic Regression model precision: 14.46%
Logistic Regression model recall: 57.64%


Linear Support Vector Classification's confusion matrix: 
True Positive(TP)  =  254
False Positive(FP) =  120
True Negative(TN)  =  45844
False Negative(FN) =  4518
Linear Support Vector Classification model accuracy: 90.86%
Linear Support Vector Classification model precision: 5.32%
Linear Support Vector Classification model recall: 67.91%


Decision Tree Classifier's confusion matrix: 
True Positive(TP)  =  1316
False Positive(FP) =  4102
True Negative(TN)  =  41862
False Negative(FN) =  3456
Decision Tree Classifier model accuracy: 85.10%
Decision Tree Classifier model precision: 27.58%
Decision Tree Classifier model recall: 24.29%


Random Forest Classifier's confusion matrix: 
True Positive(TP)  =  614
False

In [209]:
LR_2 = Classification(df_min_max_scaled_global, 'HeartDiseaseorAttack', 0.20, 0, True, None, 10000, 0)

In [210]:
LR_2.main()

Logistic Regression's confusion matrix: 
True Positive(TP)  =  689
False Positive(FP) =  510
True Negative(TN)  =  45454
False Negative(FN) =  4083
Logistic Regression model accuracy: 90.95%
Logistic Regression model precision: 14.44%
Logistic Regression model recall: 57.46%


Linear Support Vector Classification's confusion matrix: 
True Positive(TP)  =  254
False Positive(FP) =  120
True Negative(TN)  =  45844
False Negative(FN) =  4518
Linear Support Vector Classification model accuracy: 90.86%
Linear Support Vector Classification model precision: 5.32%
Linear Support Vector Classification model recall: 67.91%


Decision Tree Classifier's confusion matrix: 
True Positive(TP)  =  1354
False Positive(FP) =  4183
True Negative(TN)  =  41781
False Negative(FN) =  3418
Decision Tree Classifier model accuracy: 85.02%
Decision Tree Classifier model precision: 28.37%
Decision Tree Classifier model recall: 24.45%


Random Forest Classifier's confusion matrix: 
True Positive(TP)  =  589
False

In [211]:
LR_3 = Classification(z_scaled_global, 'HeartDiseaseorAttack', 0.20, 0, True, None, 10000, 0)

In [212]:
LR_3.main()

Logistic Regression's confusion matrix: 
True Positive(TP)  =  690
False Positive(FP) =  508
True Negative(TN)  =  45456
False Negative(FN) =  4082
Logistic Regression model accuracy: 90.95%
Logistic Regression model precision: 14.46%
Logistic Regression model recall: 57.60%


Linear Support Vector Classification's confusion matrix: 
True Positive(TP)  =  254
False Positive(FP) =  120
True Negative(TN)  =  45844
False Negative(FN) =  4518
Linear Support Vector Classification model accuracy: 90.86%
Linear Support Vector Classification model precision: 5.32%
Linear Support Vector Classification model recall: 67.91%


Decision Tree Classifier's confusion matrix: 
True Positive(TP)  =  1345
False Positive(FP) =  4143
True Negative(TN)  =  41821
False Negative(FN) =  3427
Decision Tree Classifier model accuracy: 85.08%
Decision Tree Classifier model precision: 28.19%
Decision Tree Classifier model recall: 24.51%


Random Forest Classifier's confusion matrix: 
True Positive(TP)  =  609
False